# Chain-of-Thought Generator -- Stage 0 (Analysis + Candidate Enumeration)

Variant of `cot_dataset_0.ipynb`. Instead of framing a lone equation as a prediction
task and committing to one mapping, this notebook **shows the full equation**, teaches the
basics, narrows the hypothesis space, and **enumerates 2-3 verified candidate mappings**.
There is **no `solution:` line**. See `docs/cot-template_0_examples_analysis.md`.


## 1. Setup & Imports

In [ ]:
import json
import re
from collections import Counter

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from src.encoding import encode_answer, _int_to_base_digits, _is_reversed_digit_mode
from src.operations import OPERATIONS
from src.op_index import SIGNED_OPS, CONCAT_OPS


# ---------------------------------------------------------------------------
# Equation formatting helpers (mirrors new_raw_dataset.ipynb)
# ---------------------------------------------------------------------------

def lhs_str(lhs: dict) -> str:
    return lhs["l0"] + lhs["l1"] + lhs["op"] + lhs["r0"] + lhs["r1"]


def rhs_str(rh: dict) -> str:
    body = "".join(rh["actual_rhs"])
    if rh["op"] is None:
        return body
    return rh["op"] + body if rh["sign_pos"] == "prefix" else body + rh["op"]


# ---------------------------------------------------------------------------
# JSON-decode guard (handles both string and already-decoded values)
# ---------------------------------------------------------------------------

def safe_json(v, default=None):
    if isinstance(v, str):
        return json.loads(v)
    if isinstance(v, float) and pd.isna(v):
        return default
    return v


print("Imports OK")

## 2. Load Dataset (grouped by equation `id`)

In [ ]:
PARQUET_IN  = "data/result_examples_0_result_k_6.parquet"
PARQUET_OUT = "data/result_examples_0_analysis_cot_unbalanced.parquet"

df = pq.read_table(PARQUET_IN).to_pandas()

# Decode JSON-string columns
for col in ["equations", "ops_mapping", "digits_mapping", "ops_candidates"]:
    df[col] = df[col].apply(lambda v: safe_json(v))

# Derive category from solver_strategy (the balanced parquet omits the column)
STRATEGY_TO_CATEGORY = {
    "pure_concat":   "concatenation",
    "standard":      "arithmetic",
    "little_endian": "arithmetic",
}
df["solver_category"] = df["solver_strategy"].map(STRATEGY_TO_CATEGORY)
assert df["solver_category"].notna().all()
assert df["solver_correct"].all(), "Unexpected wrong-answer rows!"
assert len(df) > 0, "Empty dataset!"

# This dataset stores one row per *candidate* solution; an equation `id` may have several.
print(f"Loaded {len(df):,} candidate rows across {df['problem_id'].nunique():,} equations")
print(df["solver_strategy"].value_counts())


## 3. Static CoT Blocks

In [ ]:
# ---------------------------------------------------------------------------
# Per-operation metadata (formula and guard in natural language)
# NOTE: maintain in lockstep with src/operations.py OPERATIONS dict ordering.
# ---------------------------------------------------------------------------

OP_FORMULA_NL = {
    "mul":        "a × b",
    "add":        "a + b",
    "absdiff":    "|a − b|",
    "sub_signed": "a − b  (result may be negative; if so, the operator symbol is attached to the answer as a sign)",
    "rsub_signed":"b − a  (same sign convention as sub_signed)",
    "neg_absdiff":"−|a − b|  (always non-positive; the operator symbol is always attached to the answer)",
    "concat_fwd": "the symbols of a followed by the symbols of b  (structural, no arithmetic)",
    "concat_rev": "the symbols of b followed by the symbols of a  (structural, no arithmetic)",
    "add_m1":     "a + b − 1  (invalid if result would be negative)",
    "add_p1":     "a + b + 1",
    "add_p2":     "a + b + 2",
    "mul_m1":     "a × b − 1  (invalid if result would be negative)",
    "mul_p1":     "a × b + 1",
    "absdiff_m1": "|a − b| − 1  (invalid when a = b)",
    "absdiff_p1": "|a − b| + 1  (always at least 1)",
    "absdiff_m2": "|a − b| − 2  (invalid when |a − b| < 2)",
    "gcd":        "greatest common divisor of a and b  (always non-negative)",
    "lcm":        "least common multiple of a and b  (0 if either operand is 0)",
    "mod":        "a mod b  (invalid when b = 0)",
    "rmod":       "b mod a  (invalid when a = 0)",
    "fdiv":       "⌊a ÷ b⌋  (invalid when b = 0)",
    "a2_plus_b":  "a² + b",
}

# Short formula for inline numeric expressions
OP_FORMULA_SHORT = {
    "mul":        "a × b",
    "add":        "a + b",
    "absdiff":    "|a − b|",
    "sub_signed": "a − b (signed)",
    "rsub_signed":"b − a (signed)",
    "neg_absdiff":"−|a − b|",
    "concat_fwd": "symbols of a + symbols of b",
    "concat_rev": "symbols of b + symbols of a",
    "add_m1":     "a + b − 1",
    "add_p1":     "a + b + 1",
    "add_p2":     "a + b + 2",
    "mul_m1":     "a × b − 1",
    "mul_p1":     "a × b + 1",
    "absdiff_m1": "|a − b| − 1",
    "absdiff_p1": "|a − b| + 1",
    "absdiff_m2": "|a − b| − 2",
    "gcd":        "gcd(a, b)",
    "lcm":        "lcm(a, b)",
    "mod":        "a mod b",
    "rmod":       "b mod a",
    "fdiv":       "⌊a ÷ b⌋",
    "a2_plus_b":  "a² + b",
}

assert set(OP_FORMULA_NL.keys()) == set(OPERATIONS.keys()), (
    "OP_FORMULA_NL is out of sync with OPERATIONS: "
    + str(set(OPERATIONS.keys()) ^ set(OP_FORMULA_NL.keys()))
)


# ---------------------------------------------------------------------------
# Build OPS_BLOCK — numbered list of all 22 ops
# ---------------------------------------------------------------------------

def make_ops_block() -> str:
    lines = ["Let's assume the following set of available operations that could govern the secret equations:\n"]
    for i, name in enumerate(OPERATIONS.keys(), 1):   # insertion order = OPERATIONS order
        formula = OP_FORMULA_NL[name]
        lines.append(f"{i:2d}. {name}(a, b) = {formula}")
    return "\n".join(lines)


OPS_BLOCK = make_ops_block()
print(OPS_BLOCK)

In [ ]:
# ---------------------------------------------------------------------------
# STRATEGIES_BLOCK — strategies + base + LHS/RHS structure + objective
# Fixed text; identical for every row.
# ---------------------------------------------------------------------------

STRATEGIES_BLOCK = """\
There are three possible solving strategies:

- Pure concatenation: the operator simply joins the two operands. The result is either the
  symbols of the left operand followed by those of the right operand (forward), or the symbols
  of the right operand followed by those of the left (reverse). No digit mapping is needed.

- Standard arithmetic (base 10, most-significant digit first): each unique symbol maps to a
  unique decimal digit (0-9). The left operand is decoded as L = digit(l0) x 10 + digit(l1),
  and the right operand as R = digit(r0) x 10 + digit(r1). The chosen operation is applied:
  result = OP(L, R). The magnitude is then converted to decimal digits from most to least
  significant, zero-padded on the left if the answer needs more digits, and each digit is
  mapped back to its symbol.

- Little-endian arithmetic (base 10, least-significant digit first): the same as standard
  arithmetic, except each operand's two digits are swapped before the operation -- the symbol
  in the first position carries the units digit (not the tens digit), so the raw two-digit
  value X is interpreted as (X mod 10) x 10 + (X / 10). The result digit sequence is
  reversed after zero-padding before mapping back to symbols.

We always use base 10.

The left-hand side of any equation has exactly 5 symbols: l0 l1 op r0 r1, where op is the
operator symbol, l0 l1 is the left operand, and r0 r1 is the right operand.

The right-hand side can have 1 to 4 symbols:
  - mag       -- all symbols represent the magnitude of the result.
  - op mag    -- operator symbol first, then magnitude (result is negative; sign prefix).
  - mag op    -- magnitude first, operator symbol last (result is negative; sign postfix).

The goal is to find the digit mapping (symbol -> digit) and operation mapping (op -> function)
that satisfies the equation, then apply them to decode the query answer."""

print(STRATEGIES_BLOCK)

In [ ]:
# ---------------------------------------------------------------------------
# STRATEGIES_BLOCK_ANALYSIS -- same as STRATEGIES_BLOCK but the closing objective
# describes analysis (which mappings are consistent) rather than predicting an answer.
# ---------------------------------------------------------------------------

_OLD_GOAL = (
    "The goal is to find the digit mapping (symbol -> digit) and operation mapping (op -> function)\n"
    "that satisfies the equation, then apply them to decode the query answer."
)
_NEW_GOAL = (
    "The goal is to understand which digit mappings (symbol -> digit), operation mappings (op ->\n"
    "function), and strategies are consistent with the equation."
)
STRATEGIES_BLOCK_ANALYSIS = STRATEGIES_BLOCK.replace(_OLD_GOAL, _NEW_GOAL)
assert STRATEGIES_BLOCK_ANALYSIS != STRATEGIES_BLOCK, "goal-paragraph replace failed"
print("STRATEGIES_BLOCK_ANALYSIS ready")


## 4. Per-Equation Helpers

In [ ]:
# ---------------------------------------------------------------------------
# Symbol frequency ordering
# Returns content symbols (non-op) ordered by descending occurrence count;
# ties broken by first appearance (stable sort).
# ---------------------------------------------------------------------------

def symbol_freq_order(eq: dict) -> list[str]:
    lhs = eq["lhs"]
    op_char = lhs["op"]
    all_chars = (
        [lhs["l0"], lhs["l1"], lhs["r0"], lhs["r1"]]
        + [c for c in eq["rhs"]["actual_rhs"] if c != op_char]
    )
    # First-appearance order
    seen: dict[str, None] = {}
    for c in all_chars:
        if c not in seen:
            seen[c] = None
    ordered = list(seen.keys())
    cnt = Counter(all_chars)
    # Stable sort by descending count keeps first-appearance order for ties
    return sorted(ordered, key=lambda s: -cnt[s])


# ---------------------------------------------------------------------------
# Numeric helpers
# ---------------------------------------------------------------------------

def decode_lhs(eq: dict, dm: dict, radix: int = 10) -> tuple[int, int]:
    lhs = eq["lhs"]
    L = dm[lhs["l0"]] * radix + dm[lhs["l1"]]
    R = dm[lhs["r0"]] * radix + dm[lhs["r1"]]
    return L, R


def swap_digits(x: int, radix: int = 10) -> int:
    return (x % radix) * radix + x // radix


def format_op_call(op_name: str, L: int, R: int) -> str:
    """Human-readable arithmetic expression, e.g. '63 x 92', '|63 - 92|'."""
    d = {
        "mul":        f"{L} x {R}",
        "add":        f"{L} + {R}",
        "absdiff":    f"|{L} - {R}|",
        "sub_signed": f"{L} - {R}",
        "rsub_signed":f"{R} - {L}",
        "neg_absdiff":f"-|{L} - {R}|",
        "add_m1":     f"{L} + {R} - 1",
        "add_p1":     f"{L} + {R} + 1",
        "add_p2":     f"{L} + {R} + 2",
        "mul_m1":     f"{L} x {R} - 1",
        "mul_p1":     f"{L} x {R} + 1",
        "absdiff_m1": f"|{L} - {R}| - 1",
        "absdiff_p1": f"|{L} - {R}| + 1",
        "absdiff_m2": f"|{L} - {R}| - 2",
        "gcd":        f"gcd({L}, {R})",
        "lcm":        f"lcm({L}, {R})",
        "mod":        f"{L} mod {R}",
        "rmod":       f"{R} mod {L}",
        "fdiv":       f"floor({L} / {R})",
        "a2_plus_b":  f"{L}^2 + {R}",
    }
    return d.get(op_name, f"{op_name}({L}, {R})")


print("Helpers defined")

In [ ]:
# ---------------------------------------------------------------------------
# OBSERVATIONS block (copied verbatim from cot_dataset_0.ipynb)
# ---------------------------------------------------------------------------

def build_observations(eq: dict, answer: str, category: str) -> str:
    lhs = eq["lhs"]
    rhs = eq["rhs"]
    op_char = lhs["op"]
    sign_pos = rhs["sign_pos"]
    actual_rhs = rhs["actual_rhs"]
    res_len = len(actual_rhs)

    bullets = []

    if sign_pos == "prefix":
        mag_str = "".join(actual_rhs)
        bullets.append(
            f"The answer {answer} starts with the operator symbol {op_char}, "
            f"indicating the result is negative (sign prefix). "
            f"The magnitude part has {res_len} symbol(s): {mag_str}."
        )
    elif sign_pos == "postfix":
        mag_str = "".join(actual_rhs)
        bullets.append(
            f"The answer {answer} ends with the operator symbol {op_char}, "
            f"indicating the result is negative (sign postfix). "
            f"The magnitude part has {res_len} symbol(s): {mag_str}."
        )
    else:
        bullets.append(
            f"The answer {answer} carries no operator symbol, so the result is "
            f"non-negative and no signed operation is in play."
        )

    if sign_pos is None:
        if res_len == 4:
            bullets.append(
                f"The answer has 4 symbols, matching the combined length of the two operands "
                f"(2 + 2). This is consistent with either a 4-digit arithmetic result or "
                f"concatenation."
            )
        else:
            bullets.append(
                f"The answer has {res_len} symbol(s) — fewer than 4, so concatenation "
                f"(which always produces 4 symbols) is already ruled out."
            )

    return "Observations:\n" + "\n".join(f"- {b}" for b in bullets)

print("Observations builder defined")


## 5. Analysis + Candidate-Enumeration Builders

In [ ]:
# ---------------------------------------------------------------------------
# ANALYSIS BUILDERS -- narrowing, per-candidate derivation, candidate selection,
# underdetermination close, prompt, and the top-level assembler.
# ---------------------------------------------------------------------------

def build_narrowing(eq: dict, answer: str) -> str:
    """Narrow the hypothesis space: rule out concatenation / wrong sign families."""
    lhs = eq["lhs"]
    op_char = lhs["op"]
    sign_pos = eq["rhs"]["sign_pos"]
    fwd = lhs["l0"] + lhs["l1"] + lhs["r0"] + lhs["r1"]
    rev = lhs["r0"] + lhs["r1"] + lhs["l0"] + lhs["l1"]

    if sign_pos == "prefix":
        return (
            f"The answer begins with the operator symbol {op_char}; concatenation cannot "
            f"produce a leading sign character, so concatenation is ruled out. Because the "
            f"answer carries a sign, the underlying result is negative; a non-negative "
            f"operation such as an absolute difference can never need a sign, so the operation "
            f"must be a signed subtraction."
        )
    if sign_pos == "postfix":
        return (
            f"The answer ends with the operator symbol {op_char}; concatenation produces only "
            f"bare operand symbols with no added sign character, so concatenation is ruled out. "
            f"The sign means the underlying result is negative, and an absolute difference is "
            f"always non-negative, so the operation must be a signed subtraction."
        )
    return (
        f"Checking concatenation: forward (l0 l1 r0 r1) gives {fwd} and reverse "
        f"(r0 r1 l0 l1) gives {rev}. Neither matches {answer}, so concatenation is ruled out. "
        f"The answer carries no sign character, so a negated difference (which always attaches "
        f"the operator symbol) is also ruled out -- the operation is arithmetic under standard "
        f"or little-endian decoding."
    )


def build_concat_blocks(eq: dict, answer: str) -> tuple[str, str]:
    """Return (narrowing_text, candidate_block) for a pure-concatenation equation."""
    lhs = eq["lhs"]
    fwd = lhs["l0"] + lhs["l1"] + lhs["r0"] + lhs["r1"]
    rev = lhs["r0"] + lhs["r1"] + lhs["l0"] + lhs["l1"]
    left = lhs["l0"] + lhs["l1"]
    right = lhs["r0"] + lhs["r1"]
    if answer == fwd:
        narrowing = (
            f"Checking concatenation: forward concatenation (left operand followed by right "
            f"operand) gives {fwd}, which matches the answer exactly. No digit mapping is needed."
        )
        cand = (
            "**Candidate -- pure concatenation (forward).**\n"
            f"The operator joins the operands left-to-right: {left} followed by {right} gives "
            f"{fwd}.\n"
            "This reproduces the shown right-hand side"
        )
    else:
        narrowing = (
            f"Checking concatenation: forward gives {fwd}, which does not match; reverse "
            f"concatenation (right operand followed by left operand) gives {rev}, which matches "
            f"the answer exactly. No digit mapping is needed."
        )
        cand = (
            "**Candidate -- pure concatenation (reverse).**\n"
            f"The operator joins the operands right-to-left: {right} followed by {left} gives "
            f"{rev}.\n"
            "This reproduces the shown right-hand side"
        )
    return narrowing, cand


def build_candidate_arith(eq: dict, dm: dict, om: dict, strategy: str, label: str) -> str:
    """Compact per-candidate derivation for one arithmetic mapping."""
    lhs = eq["lhs"]
    op_char = lhs["op"]
    op_name = om[op_char]
    actual_rhs = eq["rhs"]["actual_rhs"]
    sign_pos = eq["rhs"]["sign_pos"]
    res_len = len(actual_rhs)
    inv = {v: k for k, v in dm.items()}
    freq = symbol_freq_order(eq)

    L, R = decode_lhs(eq, dm)
    if strategy == "little_endian":
        a, b = swap_digits(L), swap_digits(R)
    else:
        a, b = L, R
    result_raw = OPERATIONS[op_name](a, b)
    mag = abs(result_raw) if (op_name in SIGNED_OPS or op_name == "neg_absdiff") else result_raw

    digits = _int_to_base_digits(mag, 10)
    needs_pad = len(digits) < res_len
    padded = [0] * (res_len - len(digits)) + digits
    seq = padded[::-1] if strategy == "little_endian" else padded
    max_val = 10 ** res_len

    strat_desc = "little-endian arithmetic" if strategy == "little_endian" else "standard arithmetic"
    lines = [f"**Candidate {label} -- {strat_desc}, {op_char} -> {op_name} ({OP_FORMULA_SHORT[op_name]}).**"]

    if strategy == "little_endian":
        lines.append(
            "In little-endian encoding each operand's two digits are read with the first symbol "
            "carrying the units digit; the result digit sequence is reversed before mapping back."
        )

    lines.append("Assign digits to the most frequent symbols first: "
                 + ", ".join(f"{s} -> {dm[s]}" for s in freq) + ".")

    if strategy == "little_endian":
        lines.append(
            f"Raw left value: {dm[lhs['l0']]} x 10 + {dm[lhs['l1']]} = {L}; "
            f"raw right value: {dm[lhs['r0']]} x 10 + {dm[lhs['r1']]} = {R}.\n"
            f"Applying the digit swap (units <-> tens): {L} -> {a}, {R} -> {b}."
        )
    else:
        lines.append(
            f"Left operand:  digit({lhs['l0']}) x 10 + digit({lhs['l1']}) = "
            f"{dm[lhs['l0']]} x 10 + {dm[lhs['l1']]} = {L}.\n"
            f"Right operand: digit({lhs['r0']}) x 10 + digit({lhs['r1']}) = "
            f"{dm[lhs['r0']]} x 10 + {dm[lhs['r1']]} = {R}."
        )

    lines.append(
        f"Applying {OP_FORMULA_SHORT[op_name]}: {format_op_call(op_name, a, b)} = {result_raw}. "
        f"The magnitude {mag} fits within {res_len} position(s) ({mag} < {max_val})."
    )

    if needs_pad:
        lines.append(
            f"The magnitude {mag} has {len(digits)} digit(s) but the answer has {res_len} "
            f"position(s), so we zero-pad on the left: {padded}."
        )
    if strategy == "little_endian":
        lines.append(f"Digits {padded} reversed for little-endian: {seq}.")

    mag_symbols = "".join(inv[d] for d in seq)
    lines.append("Converting digits back to symbols: "
                 + ", ".join(f"{d} -> {inv[d]}" for d in seq) + f" -> {mag_symbols}.")

    if sign_pos == "prefix":
        lines.append(f"The result is negative, so the operator symbol {op_char} is prepended.")
    elif sign_pos == "postfix":
        lines.append(f"The result is negative, so the operator symbol {op_char} is appended.")

    lines.append("This reproduces the shown right-hand side")
    return "\n".join(lines)


def select_candidates(cands: list[dict], k: int = 3) -> list[dict]:
    """Dedup then greedily pick up to k candidates, maximising (strategy, op) variety."""
    seen, uniq = set(), []
    for c in cands:
        key = (tuple(sorted(c["dm"].items())), c["strategy"], tuple(sorted(c["om"].items())))
        if key in seen:
            continue
        seen.add(key)
        uniq.append(c)
    if len(uniq) <= k:
        return uniq

    def sig(c):
        return (c["strategy"], tuple(sorted(c["om"].values())))

    chosen, used = [], set()
    for c in uniq:
        if sig(c) not in used:
            chosen.append(c)
            used.add(sig(c))
        if len(chosen) == k:
            return chosen
    for c in uniq:  # fill remaining slots if distinct signatures were too few
        if c not in chosen:
            chosen.append(c)
        if len(chosen) == k:
            break
    return chosen


def build_underdetermination(solution_count: int, category: str, n_shown: int) -> str:
    if category == "concatenation":
        return ("This equation is fully determined under concatenation -- only the concatenation "
                "rule is consistent, and no digit mapping is required.")
    if solution_count <= 1:
        return "This equation is fully determined -- only one mapping is consistent."
    return (
        f"A single equation does not pin down a unique mapping: the solver found "
        f"{solution_count} consistent mapping(s) here, of which the {n_shown} candidate(s) "
        f"above are examples. In a full puzzle, the shown examples narrow this set."
    )


def build_prompt(eq: dict) -> str:
    return (
        "In Alice's Wonderland, a secret set of transformation rules is applied to equations. "
        "Here is one such equation:\n"
        f"  {lhs_str(eq['lhs'])} = {rhs_str(eq['rhs'])}\n"
        "A single equation does not uniquely pin down the hidden rules. Explain what can be "
        "inferred about the operation mapping, digit mapping, and solving strategy from this "
        "equation alone, and give a few example mappings that are each consistent with it."
    )


def build_analysis_cot(group) -> tuple[str, int]:
    """Assemble one analysis CoT for an equation `id` group. Returns (cot, n_candidates)."""
    row0 = group.iloc[0]
    eq = row0["equations"][0]
    category = row0["category"]
    answer = rhs_str(eq["rhs"])
    solution_count = int(row0["solution_count"])

    sections = [
        OPS_BLOCK,
        STRATEGIES_BLOCK_ANALYSIS,
        f"Shown equation:\n  {lhs_str(eq['lhs'])} = {answer}",
        build_observations(eq, answer, category),
    ]

    if category == "concatenation":
        narrowing, cand = build_concat_blocks(eq, answer)
        sections.append(narrowing)
        sections.append(cand)
        n_shown = 1
        sections.append(build_underdetermination(solution_count, category, n_shown))
        return "\n\n".join(sections), n_shown

    cands = [
        {"dm": r["digits_mapping"], "om": r["ops_mapping"], "strategy": r["solver_strategy"]}
        for _, r in group.iterrows()
        if isinstance(r["digits_mapping"], dict) and isinstance(r["ops_mapping"], dict)
    ]
    chosen = select_candidates(cands, k=3)

    sections.append(build_narrowing(eq, answer))

    labels = "ABCDE"
    op_char = eq["lhs"]["op"]
    blocks = []
    for idx, c in enumerate(chosen):
        mode = "little_endian" if c["strategy"] == "little_endian" else "standard"
        enc = encode_answer(c["dm"], c["om"], op_char, eq["lhs"], mode, 10, answer)
        assert enc is not None and enc[0] == answer, (
            f"{row0['id']}: candidate {idx} reconstructs {enc!r} != shown rhs {answer!r}"
        )
        blocks.append(build_candidate_arith(eq, c["dm"], c["om"], c["strategy"], labels[idx]))
    sections.append("\n\n".join(blocks))

    sections.append(build_underdetermination(solution_count, category, len(chosen)))
    return "\n\n".join(sections), len(chosen)


print("Analysis builders defined")


## 6. Generate All Equations

In [ ]:
print(f"Generating analysis CoT over {df['problem_id'].nunique():,} equations...")

VOICE_FORBIDDEN = [".py", "def ", "encode_answer", "OPERATIONS", "src/", "Heuristic", "solution:"]

records = []
for gid, group in df.groupby("id", sort=False):
    cot, n_cand = build_analysis_cot(group)
    for bad in VOICE_FORBIDDEN:
        assert bad not in cot, f"{gid}: forbidden token {bad!r} leaked into cot"
    eq = group.iloc[0]["equations"][0]
    records.append({
        "id": gid,
        "prompt": build_prompt(eq),
        "cot": cot,
        "category": group.iloc[0]["category"],
        "solution_count": int(group.iloc[0]["solution_count"]),
        "n_candidates": n_cand,
        "cot_approx_tokens": len(cot) // 4,
    })

out = pd.DataFrame(records)
print(out.shape)
print(out["category"].value_counts())
print("n_candidates distribution:\n", out["n_candidates"].value_counts().sort_index())
print("approx tokens: mean", int(out["cot_approx_tokens"].mean()),
      "max", int(out["cot_approx_tokens"].max()))


## 7. Spot-Check

In [ ]:
for cat in ["arithmetic", "concatenation"]:
    sub = out[out["category"] == cat]
    if sub.empty:
        continue
    # prefer a multi-candidate example for arithmetic
    r = sub.sort_values("n_candidates", ascending=False).iloc[0]
    print("=" * 100)
    print(r["id"], "|", cat, "| candidates:", r["n_candidates"], "| solution_count:", r["solution_count"])
    print("-" * 100)
    print("PROMPT:\n" + r["prompt"])
    print("\nCOT:\n" + r["cot"])


## 8. Write Parquet

In [ ]:
assert "cot" in out.columns and out["cot"].notna().all() and (out["cot"].str.len() > 0).all(), \
    "cot column incomplete -- run the generation cell first"
assert "solution:" not in "".join(out["cot"].head(50)), "unexpected solution: line in analysis cot"

table = pa.Table.from_pandas(out, preserve_index=False)
pq.write_table(table, PARQUET_OUT)
print(f"Wrote {PARQUET_OUT}: {out.shape}")


In [ ]:
out['n_candidates'].value_counts()